# Federated Learning Tutorial

This notebook demonstrates how to use the Federated Learning system.

## Table of Contents
1. [Setup and Imports](#setup)
2. [Data Loading and Visualization](#data)
3. [Configuration](#config)
4. [Training](#training)
5. [Evaluation and Results](#evaluation)
6. [Advanced: Privacy-Preserving FL](#privacy)
7. [Advanced: Non-IID Data](#noniid)

## 1. Setup and Imports <a name="setup"></a>

First, let's import the necessary libraries and modules.

In [ ]:
import sys
from pathlib import Path

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))

import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Import FL components
from config import FLConfig, AggregationAlgorithm, DatasetType
from models import create_model
from server import CentralServer
from client import FederatedClient
from algorithms import FedAvgAggregator
from utils import NonIIDDataSplitter
from evaluation import FederatedMetrics

print("✅ All imports successful!")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 2. Data Loading and Visualization <a name="data"></a>

Let's load the MNIST dataset and visualize some samples.

In [ ]:
# Load MNIST dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(
    '../data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    '../data',
    train=False,
    download=True,
    transform=transform
)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

In [ ]:
# Visualize some samples
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    img, label = train_dataset[i]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(f'Label: {label}')
    ax.axis('off')
plt.tight_layout()
plt.show()

## 3. Configuration <a name="config"></a>

Configure the federated learning setup.

In [ ]:
# Create FL configuration
config = FLConfig()

# Basic settings
config.data.dataset = DatasetType.MNIST
config.data.num_clients = 5
config.data.iid = True  # Start with IID distribution

# Model settings
config.model.model_name = 'mnist_net'
config.model.num_classes = 10
config.model.input_channels = 1

# Training settings
config.training.num_rounds = 10
config.client.local_epochs = 2
config.client.batch_size = 32
config.client.learning_rate = 0.01

# Server settings
config.server.num_clients = 5
config.server.min_clients_per_round = 2
config.server.max_clients_per_round = 5
config.server.aggregation_algorithm = AggregationAlgorithm.FEDAVG

print("✅ Configuration created")
print(f"   - {config.data.num_clients} clients")
print(f"   - {config.training.num_rounds} rounds")
print(f"   - Algorithm: {config.server.aggregation_algorithm.value}")

### Split Data Across Clients

Distribute the training data among clients.

In [ ]:
# Split data using IID splitter
data_splitter = NonIIDDataSplitter(
    dataset=train_dataset,
    num_clients=config.data.num_clients,
    strategy='iid',
    seed=42
)
client_datasets = data_splitter.split()

# Visualize data distribution
client_sizes = [len(ds) for ds in client_datasets]
plt.figure(figsize=(10, 4))
plt.bar(range(len(client_sizes)), client_sizes)
plt.xlabel('Client ID')
plt.ylabel('Number of Samples')
plt.title('Data Distribution Across Clients (IID)')
plt.xticks(range(len(client_sizes)))
for i, size in enumerate(client_sizes):
    plt.text(i, size, str(size), ha='center', va='bottom')
plt.tight_layout()
plt.show()

print(f"✅ Data split across {len(client_datasets)} clients")
print(f"   Mean samples per client: {np.mean(client_sizes):.0f}")
print(f"   Std samples per client: {np.std(client_sizes):.2f}")

## 4. Training <a name="training"></a>

Set up and run federated training.

In [ ]:
# Create global model
global_model = create_model(
    config.model.model_name,
    num_classes=config.model.num_classes
)

print(f"✅ Model created: {config.model.model_name}")
print(f"   Total parameters: {sum(p.numel() for p in global_model.parameters()):,}")

In [ ]:
# Create aggregator and server
aggregator = FedAvgAggregator(device='cpu')

server = CentralServer(
    model=global_model,
    aggregator=aggregator,
    num_clients=config.server.num_clients,
    device='cpu',
    min_clients_per_round=config.server.min_clients_per_round,
    max_clients_per_round=config.server.max_clients_per_round,
    client_selection_strategy='random'
)

# Set test data
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
server.set_test_data(test_loader)

print("✅ Server initialized")

In [ ]:
# Create clients
clients = []
for client_id, train_data in enumerate(client_datasets):
    model = create_model(config.model.model_name, num_classes=config.model.num_classes)
    client = FederatedClient(
        client_id=client_id,
        model=model,
        train_data=train_data,
        device='cpu',
        local_epochs=config.client.local_epochs,
        batch_size=config.client.batch_size,
        learning_rate=config.client.learning_rate
    )
    clients.append(client)

print(f"✅ Created {len(clients)} clients")

In [ ]:
# Training loop
metrics = FederatedMetrics()
test_accuracies = []
test_losses = []

print("\n🚀 Starting Federated Training\n")

for round_num in range(config.training.num_rounds):
    print(f"Round {round_num + 1}/{config.training.num_rounds}")
    
    # Execute federated round
    round_stats = server.federated_round(clients)
    
    # Track metrics
    test_accuracies.append(round_stats['global_test_accuracy'])
    test_losses.append(round_stats['global_test_loss'])
    
    metrics.update(
        round_num=round_num,
        global_metrics={
            'test_accuracy': round_stats['global_test_accuracy'],
            'test_loss': round_stats['global_test_loss']
        },
        client_metrics=round_stats.get('client_metrics')
    )
    
    print(f"  Test Acc: {round_stats['global_test_accuracy']:.2f}% | Test Loss: {round_stats['global_test_loss']:.4f}")

print("\n✅ Training Complete!")

## 5. Evaluation and Results <a name="evaluation"></a>

Visualize training results and metrics.

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Accuracy
ax1.plot(range(1, len(test_accuracies) + 1), test_accuracies, marker='o', linewidth=2)
ax1.set_xlabel('Round')
ax1.set_ylabel('Test Accuracy (%)')
ax1.set_title('Test Accuracy Over Rounds')
ax1.grid(True, alpha=0.3)

# Loss
ax2.plot(range(1, len(test_losses) + 1), test_losses, marker='o', linewidth=2, color='orange')
ax2.set_xlabel('Round')
ax2.set_ylabel('Test Loss')
ax2.set_title('Test Loss Over Rounds')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n📊 Final Results:")
print(f"   Best Accuracy: {max(test_accuracies):.2f}%")
print(f"   Final Accuracy: {test_accuracies[-1]:.2f}%")
print(f"   Final Loss: {test_losses[-1]:.4f}")

In [ ]:
# Print detailed metrics summary
metrics.print_summary()

## 6. Advanced: Privacy-Preserving FL <a name="privacy"></a>

Enable differential privacy for privacy-preserving federated learning.

In [ ]:
from privacy import PrivacyEngine

# Configure privacy
config.privacy.enable_privacy = True
config.privacy.epsilon = 1.0
config.privacy.delta = 1e-5
config.privacy.max_grad_norm = 1.0

# Create privacy engine
privacy_engine = PrivacyEngine(
    epsilon=config.privacy.epsilon,
    delta=config.privacy.delta,
    max_grad_norm=config.privacy.max_grad_norm,
    sample_rate=config.server.max_clients_per_round / config.server.num_clients
)

print("✅ Privacy Engine Initialized")
print(f"   ε (epsilon): {config.privacy.epsilon}")
print(f"   δ (delta): {config.privacy.delta}")
print(f"   Max gradient norm: {config.privacy.max_grad_norm}")
print("\nℹ️  Now you can train with differential privacy by passing privacy_engine to the server.")

## 7. Advanced: Non-IID Data <a name="noniid"></a>

Simulate realistic non-IID data distribution using Dirichlet distribution.

In [ ]:
# Create non-IID data split using Dirichlet
noniid_splitter = NonIIDDataSplitter(
    dataset=train_dataset,
    num_clients=5,
    strategy='dirichlet',
    alpha=0.5,  # Lower alpha = more heterogeneous
    seed=42
)
noniid_datasets = noniid_splitter.split()

# Analyze label distribution per client
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for client_id, (dataset, ax) in enumerate(zip(noniid_datasets, axes)):
    labels = [dataset[i][1] for i in range(len(dataset))]
    label_counts = np.bincount(labels, minlength=10)
    
    ax.bar(range(10), label_counts)
    ax.set_title(f'Client {client_id}')
    ax.set_xlabel('Class')
    ax.set_ylabel('Count')
    ax.set_xticks(range(10))

plt.suptitle('Non-IID Label Distribution (Dirichlet α=0.5)', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

print("✅ Non-IID data distribution created")
print("   Each client has a different distribution of labels")
print("   Lower α values create more heterogeneity")

## Summary

This tutorial covered:

1. ✅ Setting up the federated learning environment
2. ✅ Loading and visualizing data
3. ✅ Configuring FL parameters
4. ✅ Training a federated model
5. ✅ Evaluating and visualizing results
6. ✅ Advanced: Privacy-preserving FL with differential privacy
7. ✅ Advanced: Non-IID data distribution

### Next Steps

- Try different algorithms: FedProx, FedNova
- Experiment with different datasets: CIFAR-10
- Adjust hyperparameters for better performance
- Train with privacy budgets and analyze privacy-utility tradeoffs
- Compare IID vs Non-IID performance